# Herramienta 01 - Predicción de demanda de transporte

Este notebook funciona como una herramienta completa para estimar demanda por ruta. Incluye preparación de datos, análisis exploratorio, entrenamiento, evaluación y un pronóstico operativo de 30 días.


## 1. Configuración
Se cargan las librerías necesarias. El notebook no requiere clonar el repositorio: lee el CSV completo `data/processed/cta_bus_ridership_daily_by_route.csv`, tomado del Chicago Data Portal. El archivo completo queda en GitHub y el notebook filtra desde 2021 para entrenar con datos recientes.


In [ ]:
# Librerías principales
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

SEED = 42
np.random.seed(SEED)

DATA_URL = 'https://raw.githubusercontent.com/AndresGuido9820/sistema-transporte-inteligente/main/data/processed/cta_bus_ridership_daily_by_route.csv'
TRAIN_START_DATE = '2021-01-01'
TOP_RUTAS = 10


## 2. Carga y filtrado del dataset real
El archivo se carga completo desde el repositorio público. Contiene demanda diaria por ruta de buses CTA desde 2001 hasta 2026. Para entrenar un modelo operativo y rápido en Colab, se filtra desde 2021 y se toman las rutas con mayor demanda reciente.


In [ ]:
raw_demanda = pd.read_csv(DATA_URL)
raw_demanda['date'] = pd.to_datetime(raw_demanda['date'], errors='coerce')
raw_demanda['rides'] = pd.to_numeric(raw_demanda['rides'], errors='coerce')
raw_demanda = raw_demanda.dropna(subset=['date', 'route', 'rides'])

demanda = raw_demanda[raw_demanda['date'] >= pd.Timestamp(TRAIN_START_DATE)].copy()
rutas_principales = demanda.groupby('route')['rides'].sum().nlargest(TOP_RUTAS).index
demanda = demanda[demanda['route'].isin(rutas_principales)].copy()
demanda = demanda.rename(columns={'rides': 'passengers'})
demanda['holiday'] = demanda['daytype'].eq('U').astype(int)
demanda = demanda[['date', 'route', 'passengers', 'holiday', 'daytype']].sort_values(['route', 'date']).reset_index(drop=True)

print('Archivo completo cargado:', raw_demanda.shape)
print('Rango completo:', raw_demanda['date'].min().date(), 'a', raw_demanda['date'].max().date())
print('Datos usados para entrenamiento:', demanda.shape)
print('Rutas seleccionadas:', sorted(demanda['route'].unique()))
demanda.head()


## 3. Exploración inicial
Se revisa el volumen por ruta y el comportamiento temporal para detectar tendencia, estacionalidad semanal e intermitencia.


In [ ]:
resumen = demanda.groupby('route')['passengers'].agg(['count', 'mean', 'min', 'max']).round(2)
display(resumen)

plt.figure(figsize=(12, 5))
for ruta, datos in demanda.groupby('route'):
    serie = datos.sort_values('date').set_index('date')['passengers'].rolling(7).mean()
    plt.plot(serie.index, serie.values, label=ruta)
plt.title('Media móvil de 7 días por ruta')
plt.xlabel('Fecha')
plt.ylabel('Pasajeros')
plt.legend(fontsize=8)
plt.grid(alpha=0.25)
plt.show()


## 3.5 Análisis de estacionalidad y tendencias

Se analizan tres dimensiones de la demanda: **tendencia** de largo plazo (descomposición aditiva con `seasonal_decompose`, período 7 días), **patrón semanal** (demanda promedio por día de semana) y **evolución anual** (comparativa año a año por ruta). Esto permite identificar si la demanda crece o cae en el tiempo, en qué días se concentra y cómo se comporta la estacionalidad semanal.

In [ ]:
from statsmodels.tsa.seasonal import seasonal_decompose

# Ruta con mayor demanda total como referencia
ruta_ref = demanda.groupby('route')['passengers'].sum().idxmax()
serie_ref = (
    demanda[demanda['route'] == ruta_ref]
    .sort_values('date')
    .set_index('date')['passengers']
    .asfreq('D')
    .fillna(method='ffill')
)

# --- 1. Descomposición aditiva (tendencia + estacionalidad + residuo) ---
decomp = seasonal_decompose(serie_ref, model='additive', period=7)
fig, axes = plt.subplots(4, 1, figsize=(12, 9), sharex=True)
serie_ref.plot(ax=axes[0], lw=1, color='#1f5eff', title='Original')
decomp.trend.plot(ax=axes[1], lw=1.5, color='#a86200', title='Tendencia')
decomp.seasonal.plot(ax=axes[2], lw=1, color='#0a7a5f', title='Estacionalidad semanal')
decomp.resid.plot(ax=axes[3], lw=1, color='#c0392b', title='Residuo')
for ax in axes:
    ax.grid(alpha=0.25)
    ax.set_xlabel('')
plt.suptitle(f'Descomposición aditiva — Ruta {ruta_ref}', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

# --- 2. Patrón semanal promedio (todas las rutas) ---
dias = ['Lun', 'Mar', 'Mié', 'Jue', 'Vie', 'Sáb', 'Dom']
tmp = demanda.copy()
tmp['dayofweek'] = tmp['date'].dt.dayofweek
patron_semanal = tmp.groupby('dayofweek')['passengers'].mean()

plt.figure(figsize=(8, 3.5))
colores = ['#1f5eff' if i < 5 else '#f6a531' for i in range(7)]
plt.bar(dias, patron_semanal.values, color=colores)
plt.title('Demanda promedio por día de semana (todas las rutas, 2021–2026)')
plt.ylabel('Pasajeros promedio')
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

# --- 3. Evolución anual por ruta ---
tmp['year'] = tmp['date'].dt.year
anual = tmp.groupby(['year', 'route'])['passengers'].mean().reset_index()

plt.figure(figsize=(11, 4))
for ruta, datos in anual.groupby('route'):
    plt.plot(datos['year'], datos['passengers'], marker='o', label=str(ruta), lw=1.6)
plt.title('Evolución anual — promedio diario de pasajeros por ruta')
plt.xlabel('Año')
plt.ylabel('Pasajeros promedio/día')
plt.legend(fontsize=8, ncol=2)
plt.grid(alpha=0.25)
plt.xticks(sorted(tmp['year'].unique()))
plt.tight_layout()
plt.show()

## 4. Ingeniería de variables
El modelo usa variables de calendario, rezagos y media móvil. Estas variables convierten la serie de tiempo en un problema supervisado de regresión.


In [ ]:
def crear_variables(df):
    data = df.copy()
    data['date'] = pd.to_datetime(data['date'])
    data = data.sort_values(['route', 'date'])
    data['dayofweek'] = data['date'].dt.dayofweek
    data['month'] = data['date'].dt.month
    data['is_weekend'] = data['dayofweek'].isin([5, 6]).astype(int)
    data['holiday'] = (data['dayofweek'] == 6).astype(int)
    data['lag_1'] = data.groupby('route')['passengers'].shift(1)
    data['lag_7'] = data.groupby('route')['passengers'].shift(7)
    data['rolling_7'] = data.groupby('route')['passengers'].transform(lambda s: s.shift(1).rolling(7, min_periods=1).mean())
    return data.dropna().reset_index(drop=True)

features = ['dayofweek', 'month', 'is_weekend', 'holiday', 'lag_1', 'lag_7', 'rolling_7']
model_data = crear_variables(demanda)
model_data.head()


## 5. Entrenamiento y evaluación
Se comparan dos modelos por ruta. La partición respeta el orden temporal: 80% entrenamiento y 20% prueba.


In [ ]:
modelos = {
    'Random Forest': RandomForestRegressor(n_estimators=160, min_samples_leaf=2, random_state=SEED),
    'Gradient Boosting': GradientBoostingRegressor(random_state=SEED),
}

resultados = []
mejores_modelos = {}
predicciones = []

for ruta, datos_ruta in model_data.groupby('route'):
    datos_ruta = datos_ruta.sort_values('date')
    corte = int(len(datos_ruta) * 0.8)
    train, test = datos_ruta.iloc[:corte], datos_ruta.iloc[corte:]
    mejor_rmse = float('inf')
    for nombre, modelo in modelos.items():
        modelo.fit(train[features], train['passengers'])
        pred = modelo.predict(test[features])
        mae = mean_absolute_error(test['passengers'], pred)
        rmse = np.sqrt(mean_squared_error(test['passengers'], pred))
        mape = np.mean(np.abs((test['passengers'] - pred) / test['passengers'])) * 100
        resultados.append({'route': ruta, 'model': nombre, 'MAE': mae, 'RMSE': rmse, 'MAPE': mape})
        if rmse < mejor_rmse:
            mejor_rmse = rmse
            mejores_modelos[ruta] = modelo
            tmp = test[['date', 'route', 'passengers']].copy()
            tmp['prediction'] = pred
            predicciones.append(tmp)

metricas = pd.DataFrame(resultados).sort_values(['route', 'RMSE'])
display(metricas.round(3))


## 6. Gráficas de validación
Estas gráficas permiten explicar si el modelo sigue la forma de la demanda real o si se queda corto en picos específicos.


In [ ]:
pred_df = pd.concat(predicciones, ignore_index=True)
for ruta in pred_df['route'].unique()[:4]:
    graf = pred_df[pred_df['route'] == ruta].sort_values('date')
    plt.figure(figsize=(11, 4))
    plt.plot(graf['date'], graf['passengers'], marker='o', label='Real')
    plt.plot(graf['date'], graf['prediction'], marker='o', label='Predicción')
    plt.title(f'Demanda real vs. predicha - {ruta}')
    plt.ylabel('Pasajeros')
    plt.xticks(rotation=45, ha='right')
    plt.grid(alpha=0.25)
    plt.legend()
    plt.show()


## 7. Herramienta operativa: pronóstico a 30 días
Seleccione una ruta y ejecute la celda para obtener la predicción diaria futura. Esta es la salida que usaría el área de planeación para asignar vehículos y personal.


In [ ]:
#@title Parámetros de la herramienta
ruta_seleccionada = '66' #@param {type:'string'}
dias_a_predecir = 30 #@param {type:'integer'}

if ruta_seleccionada not in mejores_modelos:
    ruta_seleccionada = list(mejores_modelos.keys())[0]
    print('Ruta no encontrada. Se usará:', ruta_seleccionada)

def pronosticar_ruta(df, ruta, modelo, dias=30):
    hist = df[df['route'] == ruta].sort_values('date')[['date', 'passengers']].copy()
    valores = hist['passengers'].astype(float).tolist()
    ultima_fecha = hist['date'].max()
    filas = []
    for paso in range(1, dias + 1):
        fecha = ultima_fecha + pd.Timedelta(days=paso)
        fila = pd.DataFrame([{
            'dayofweek': fecha.dayofweek,
            'month': fecha.month,
            'is_weekend': int(fecha.dayofweek >= 5),
            'holiday': int(fecha.dayofweek == 6),
            'lag_1': valores[-1],
            'lag_7': valores[-7] if len(valores) >= 7 else valores[-1],
            'rolling_7': np.mean(valores[-7:]),
        }])[features]
        pred = max(float(modelo.predict(fila)[0]), 0)
        valores.append(pred)
        filas.append({'date': fecha.date(), 'route': ruta, 'forecast_passengers': round(pred, 2)})
    return pd.DataFrame(filas)

pronostico = pronosticar_ruta(demanda, ruta_seleccionada, mejores_modelos[ruta_seleccionada], dias_a_predecir)
display(pronostico.head(10))

plt.figure(figsize=(11, 4))
plt.plot(pronostico['date'], pronostico['forecast_passengers'], marker='o')
plt.title(f'Pronóstico de demanda - {ruta_seleccionada}')
plt.ylabel('Pasajeros estimados')
plt.xticks(rotation=45, ha='right')
plt.grid(alpha=0.25)
plt.show()


## 8. Conclusiones

### Dataset y exploración
El dataset del CTA contiene 1 106 531 registros diarios por ruta (2001–2026).
Filtrado desde 2021, las 10 rutas de mayor demanda aportan 19 160 observaciones
de entrenamiento. Las rutas con mayor demanda promedio diaria son la 66
(~12 965 pasajeros/día) y la 79 (~12 577 pasajeros/día). La descomposición
aditiva confirmó una **estacionalidad semanal clara** (menor demanda sábado y
marcada caída el domingo) y una **tendencia de recuperación post-COVID** visible
desde 2021, aunque sin recuperar los niveles previos a 2019.

### Rendimiento de los modelos
Ambos modelos (Random Forest y Gradient Boosting) son competitivos sobre series
de tiempo con ingeniería de variables de calendario y rezagos. El **MAPE global
oscila entre 7.25 % y 12.81 %** según la ruta:

| Rango MAPE | Rutas | Observación |
|---|---|---|
| 7.25 % – 8.00 % | 66, 77, 9, 22, 49 | Demanda estable, patrón semanal regular |
| 8.00 % – 9.60 % | 3, 4, 8, 79 | Mayor variabilidad intradiaria |
| > 11 % | 53 | Alta amplitud entre días laborales y festivos |

Random Forest ganó en 6 de 10 rutas; Gradient Boosting ganó en las 4 restantes
(49, 53, 77, 79). No hay un modelo dominante universal, lo que confirma que la
selección por menor RMSE por ruta es la estrategia correcta.

### Variables más informativas
Los rezagos `lag_1` y `lag_7` y la media móvil `rolling_7` capturan la
autocorrelación de corto plazo y el efecto día-de-semana equivalente de la
semana anterior. Las variables de calendario (`dayofweek`, `is_weekend`) explican
la caída sistemática de fines de semana. Estas siete variables son suficientes
para un modelo base operativo.

### Pronóstico a 30 días
El pronóstico para la ruta 22 (abril 2026) estima ~13 800–14 100 pasajeros en
días hábiles y ~10 900 en fin de semana, coherente con el promedio histórico
reciente de la ruta. El modelo propaga el patrón semanal aprendido sin
divergencia visible en el horizonte de 30 días.

### Limitaciones y mejoras sugeridas
- El modelo no incorpora **variables exógenas** (clima, festivos oficiales,
  eventos, interrupciones de servicio), que explican picos de residuo elevados.
- La ruta 53 presenta el MAPE más alto (≥ 11 %); requiere inspección de outliers
  o un modelo con mayor capacidad de regularización.
- Para producción se recomienda **re-entrenamiento incremental** (ventana
  deslizante) conforme llegan datos nuevos, dado que el patrón post-pandemia
  aún no se ha estabilizado completamente.
- Un baseline naive (predicción = mismo día de la semana anterior) permitiría
  cuantificar la ganancia real del modelo frente a la alternativa más simple.

---

## Fuente del dataset

**Chicago Transit Authority (CTA) — CTA Bus Ridership Daily Totals by Route**

- **Portal:** [Chicago Data Portal](https://data.cityofchicago.org/Transportation/CTA-Ridership-Bus-Routes-Daily-Totals-by-Route/jyb9-n7fm)
- **Publicado por:** City of Chicago — Chicago Transit Authority
- **Licencia:** Public Domain (U.S. Government Open Data)
- **Cobertura temporal:** 2001-01-01 al presente (actualización continua)
- **Granularidad:** Demanda diaria por ruta de bus, con tipo de día (`W` = laboral, `A` = sábado, `U` = domingo/festivo)
- **Tamaño descargado:** 1 106 531 filas (al 2026-03-31)